In [ ]:
# ============================================================
# C. elegans LINEAGE — SETUP
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import pickle
import requests
from collections import defaultdict, OrderedDict

BASE_DIR = Path("/content/Celegans_Lineage")
RAW_DIR = BASE_DIR / "RAW_WORMATLAS"
DATA_DIR = BASE_DIR / "DATA"
EXPORT_DIR = BASE_DIR / "EXPORT"

for d in [BASE_DIR, RAW_DIR, DATA_DIR, EXPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("RAW:", RAW_DIR)
print("DATA:", DATA_DIR)
print("EXPORT:", EXPORT_DIR)

In [ ]:
# ============================================================
# WORMLINEAGE — SOURCE REGISTER
# ============================================================

SOURCES = {
    "cell_lineages": "https://wormatlas.cropsci.illinois.edu/celllineages.html",
    "cell_list": "https://wormatlas.cropsci.illinois.edu/celllistsulston.htm",
    "embryonic_lineage": "https://wormatlas.org/SulstonembCellLin_1983/SulstonembCellLin1983.html",
    "neuronal_lineages": "https://wormatlas.org/SulstonNeuronalCellLineages/Sulston1983.html",
    "postembryonic_lineages": "https://wormatlas.cropsci.illinois.edu/SulstonpostembCell_1977/SulstonpostembCell1977.html",
}

source_table = pd.DataFrame(
    [{"name": k, "url": v} for k, v in SOURCES.items()]
)

display(source_table)

In [ ]:
# ============================================================
# DOWNLOAD WORMLINEAGE SOURCE MATERIAL
# SSL FALLBACK FOR WORMLINEAGE
# ============================================================

import urllib3

urllib3.disable_warnings(
    urllib3.exceptions.InsecureRequestWarning
)

def download_text(url, filename):
    path = RAW_DIR / filename

    r = requests.get(
        url,
        timeout=60,
        verify=False
    )
    r.raise_for_status()

    path.write_text(r.text, encoding="utf-8")

    print(f"OK  {filename:35s} {len(r.text):,} bytes")
    return path


download_text(
    SOURCES["cell_lineages"],
    "celllineages.html"
)

download_text(
    SOURCES["cell_list"],
    "celllistsulston.htm"
)

download_text(
    SOURCES["embryonic_lineage"],
    "SulstonembCellLin1983.html"
)

download_text(
    SOURCES["neuronal_lineages"],
    "Sulston1983.html"
)

download_text(
    SOURCES["postembryonic_lineages"],
    "SulstonpostembCell1977.html"
)

## Tandem-repeat (TR) acquisition

This section retrieves the *C. elegans* N2 reference genome (WBcel235), extracts the six nuclear chromosomes, runs Tandem Repeats Finder (TRF) 4.09.1, and consolidates the unfiltered results into `TR_CATALOG_RAW.tsv`.

The TRF parameters are `2 7 7 80 10 50 2000`. The mitochondrial record is excluded. No biological or sequence-based filtering and no deduplication are applied here.


In [ ]:
# ============================================================
# TR ACQUISITION — SETUP AND DOWNLOAD OF THE WBcel235 GENOME
# ============================================================

from pathlib import Path
import urllib.request
import gzip
import os

TR_DIR = Path("/content/NEW_TR")
FASTA_DIR = TR_DIR / "FASTA_CHROMOSOMES"
TRF_DIR = TR_DIR / "TRF_4.09.1"
RAW_DIR = TR_DIR / "TRF_RAW"

for folder in [TR_DIR, FASTA_DIR, TRF_DIR, RAW_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

GENOME_FILE = TR_DIR / "GCF_000002985.6_WBcel235_genomic.fna.gz"
GENOME_URL = (
    "https://ftp.ncbi.nlm.nih.gov/genomes/all/"
    "GCF/000/002/985/GCF_000002985.6_WBcel235/"
    "GCF_000002985.6_WBcel235_genomic.fna.gz"
)

if not GENOME_FILE.is_file():
    request = urllib.request.Request(
        GENOME_URL,
        headers={"User-Agent": "C-elegans-TR-research-notebook"}
    )
    print("Downloading the WBcel235 reference genome from NCBI...")
    with urllib.request.urlopen(request, timeout=120) as response:
        with GENOME_FILE.open("wb") as destination:
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                destination.write(block)
else:
    print("The WBcel235 genome file is already present.")

print("Genome file:", GENOME_FILE)
print("Compressed size:", f"{GENOME_FILE.stat().st_size:,}", "bytes")

# Read the complete gzip stream to verify its integrity.
decompressed_bytes = 0
with gzip.open(GENOME_FILE, "rb") as source:
    while True:
        block = source.read(1024 * 1024)
        if not block:
            break
        decompressed_bytes += len(block)

print("Decompressed size:", f"{decompressed_bytes:,}", "bytes")
print("GZIP integrity check: passed")


In [ ]:
# ============================================================
# TR ACQUISITION — EXTRACTION OF THE SIX NUCLEAR CHROMOSOMES
# ============================================================

import gzip

CHROMOSOMES = {
    "I":   "NC_003279.8",
    "II":  "NC_003280.10",
    "III": "NC_003281.10",
    "IV":  "NC_003282.8",
    "V":   "NC_003283.11",
    "X":   "NC_003284.9",
}

genome_sequences = {}
current_accession = None
current_sequence = []

with gzip.open(GENOME_FILE, "rt") as fasta:
    for line in fasta:
        line = line.strip()
        if not line:
            continue

        if line.startswith(">"):
            if current_accession in CHROMOSOMES.values():
                chromosome = next(
                    key for key, accession in CHROMOSOMES.items()
                    if accession == current_accession
                )
                genome_sequences[chromosome] = "".join(current_sequence).upper()

            current_accession = line[1:].split()[0]
            current_sequence = []
        elif current_accession in CHROMOSOMES.values():
            current_sequence.append(line)

if current_accession in CHROMOSOMES.values():
    chromosome = next(
        key for key, accession in CHROMOSOMES.items()
        if accession == current_accession
    )
    genome_sequences[chromosome] = "".join(current_sequence).upper()

if set(genome_sequences) != set(CHROMOSOMES):
    raise RuntimeError(
        "The six expected nuclear chromosomes were not all recovered. "
        f"Found: {sorted(genome_sequences)}"
    )

for chromosome, accession in CHROMOSOMES.items():
    sequence = genome_sequences[chromosome]
    fasta_path = FASTA_DIR / f"chr{chromosome}.fa"

    with fasta_path.open("w", encoding="utf-8") as output:
        output.write(f">{accession} Caenorhabditis elegans chromosome {chromosome}\n")
        for start in range(0, len(sequence), 80):
            output.write(sequence[start:start + 80] + "\n")

    print(
        f"Chromosome {chromosome}: {len(sequence):,} bp | "
        f"{fasta_path}"
    )

print("Six nuclear chromosome FASTA files created.")
print("The mitochondrial sequence was excluded.")


In [ ]:
# ============================================================
# TR ACQUISITION — DOWNLOAD AND VERIFY TRF 4.09.1
# ============================================================

import subprocess
import urllib.request

TRF_EXECUTABLE = TRF_DIR / "trf409.linux64"
TRF_URL = (
    "https://github.com/Benson-Genomics-Lab/TRF/releases/"
    "download/v4.09.1/trf409.linux64"
)

if not TRF_EXECUTABLE.is_file():
    print("Downloading TRF 4.09.1...")
    urllib.request.urlretrieve(TRF_URL, TRF_EXECUTABLE)
    TRF_EXECUTABLE.chmod(0o755)
else:
    print("TRF 4.09.1 executable is already present.")

TRF_EXECUTABLE.chmod(0o755)
version_check = subprocess.run(
    [str(TRF_EXECUTABLE)],
    capture_output=True,
    text=True
)

print("Executable:", TRF_EXECUTABLE)
print("TRF invocation completed; exit code:", version_check.returncode)
if version_check.stdout:
    print(version_check.stdout[:1000])
if version_check.stderr:
    print(version_check.stderr[:1000])


In [ ]:
# ============================================================
# TR ACQUISITION — TRF SCAN OF ALL SIX NUCLEAR CHROMOSOMES
# ============================================================

import subprocess

TRF_PARAMETERS = ["2", "7", "7", "80", "10", "50", "2000"]
chromosome_order = ["I", "II", "III", "IV", "V", "X"]

for chromosome in chromosome_order:
    fasta_file = FASTA_DIR / f"chr{chromosome}.fa"
    output_file = RAW_DIR / (
        f"chr{chromosome}.fa.2.7.7.80.10.50.2000.dat"
    )

    if not fasta_file.is_file():
        raise FileNotFoundError(f"Chromosome FASTA not found: {fasta_file}")

    if output_file.is_file() and output_file.stat().st_size > 0:
        print(f"Chromosome {chromosome}: existing TRF output retained.")
        continue

    command = [
        str(TRF_EXECUTABLE),
        str(fasta_file),
        *TRF_PARAMETERS,
        "-d",
        "-h"
    ]

    print(f"Running TRF on chromosome {chromosome}...")
    result = subprocess.run(
        command,
        cwd=RAW_DIR,
        capture_output=True,
        text=True
    )

    if not output_file.is_file() or output_file.stat().st_size == 0:
        raise RuntimeError(
            f"TRF output missing or empty for chromosome {chromosome}. "
            f"Exit code: {result.returncode}\n{result.stderr}"
        )

    print(
        f"Chromosome {chromosome}: output verified "
        f"({output_file.stat().st_size:,} bytes)."
    )

print("TRF scan completed for all six nuclear chromosomes.")


In [ ]:
# ============================================================
# TR ACQUISITION — CONSOLIDATION OF THE RAW TRF CATALOG
# ============================================================

import csv

CATALOG_FILE = TR_DIR / "TR_CATALOG_RAW.tsv"

catalog_columns = [
    "Chromosome", "Start", "End", "Period", "Copy_Number",
    "Consensus_Size", "Percent_Matches", "Percent_Indels",
    "Score", "A", "C", "G", "T", "Entropy",
    "Consensus", "Repeat_Sequence"
]

total_results = 0
results_by_chromosome = {}

with CATALOG_FILE.open("w", newline="", encoding="utf-8") as catalog:
    writer = csv.writer(catalog, delimiter="\t")
    writer.writerow(catalog_columns)

    for chromosome in chromosome_order:
        dat_file = RAW_DIR / (
            f"chr{chromosome}.fa.2.7.7.80.10.50.2000.dat"
        )
        if not dat_file.is_file():
            raise FileNotFoundError(f"TRF result file missing: {dat_file}")

        count = 0
        with dat_file.open("r", encoding="utf-8", errors="replace") as source:
            for line in source:
                fields = line.split()
                if (
                    len(fields) == 15
                    and fields[0].isdigit()
                    and fields[1].isdigit()
                ):
                    writer.writerow([chromosome] + fields)
                    count += 1

        if count == 0:
            raise ValueError(
                f"No valid TRF result rows recognized in {dat_file.name}"
            )

        results_by_chromosome[chromosome] = count
        total_results += count
        print(f"Chromosome {chromosome}: {count:,} TR records")

print("\nTotal raw TR records:", f"{total_results:,}")
print("Catalog:", CATALOG_FILE)
print("Catalog size:", f"{CATALOG_FILE.stat().st_size:,}", "bytes")

# This check records the count validated in the original TRF run.
if total_results != 39098:
    raise ValueError(
        "The result count differs from the previously validated catalog "
        f"(39,098): found {total_results:,}. Review the outputs before "
        "treating this catalog as equivalent."
    )

print("Validated raw catalog count: 39,098 records.")
print("No biological filtering or deduplication has been applied.")


In [ ]:
# ============================================================
# TR ACQUISITION — ARCHIVE THE RAW RESULTS
# ============================================================

import zipfile

ARCHIVE_FILE = TR_DIR / "TRF_RAW_ARCHIVE.zip"

files_to_archive = [
    CATALOG_FILE,
    GENOME_FILE,
    *[
        RAW_DIR / f"chr{chromosome}.fa.2.7.7.80.10.50.2000.dat"
        for chromosome in chromosome_order
    ],
]

missing_files = [str(path) for path in files_to_archive if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Cannot create the archive; files are missing:\n"
        + "\n".join(missing_files)
    )

manifest = TR_DIR / "README_RAW_TR_DATA.txt"
manifest.write_text(
    "RAW TANDEM-REPEAT DATA — Caenorhabditis elegans N2\n"
    "Reference assembly: GCF_000002985.6_WBcel235\n"
    "Nuclear chromosomes: I, II, III, IV, V, X\n"
    "Mitochondrial sequence: excluded\n"
    "Software: Tandem Repeats Finder (TRF) 4.09.1\n"
    "Parameters: 2 7 7 80 10 50 2000\n"
    "Catalog: TR_CATALOG_RAW.tsv\n"
    "Filtering: none\n"
    "Deduplication: none\n",
    encoding="utf-8"
)

with zipfile.ZipFile(
    ARCHIVE_FILE, "w", compression=zipfile.ZIP_DEFLATED
) as archive:
    for path in files_to_archive:
        archive.write(path, arcname=path.name)
    archive.write(manifest, arcname=manifest.name)

print("Raw-data archive created:", ARCHIVE_FILE)
print("Archive size:", f"{ARCHIVE_FILE.stat().st_size:,}", "bytes")
print("Files included:", len(files_to_archive) + 1)


In [ ]:
# ============================================================
# TR ACQUISITION — DOWNLOAD THE RAW-DATA ARCHIVE
# ============================================================

from google.colab import files

if not ARCHIVE_FILE.is_file():
    raise FileNotFoundError(f"Archive not found: {ARCHIVE_FILE}")

print("Ready to download:", ARCHIVE_FILE)
print("Size:", f"{ARCHIVE_FILE.stat().st_size:,}", "bytes")
files.download(str(ARCHIVE_FILE))
